# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading and basic exploration of the FAIR^2 dataset on protein abundance, modifications, and sequences in human (Homo sapiens) samples using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, their associated fields, and the corresponding `@id`s. This is important for referencing data later.

We use the metadata to enumerate all record sets and the fields for each.

In [ ]:
# List all record sets and their fields by @id
import pprint
record_sets = metadata.record_sets

print("Record sets and their fields by @id:\n")
for rs in record_sets:
    print(f"Record set @id: {rs._id}\n  Name: {getattr(rs, 'name', '[None]')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field._id}    (name: {getattr(field, 'name', '[None]')})")
    print()

## 3. Data Extraction
Select a record set by its `@id` for exploration and load it into a pandas DataFrame. This will allow you to analyze the structure and contents of the data.

We'll extract all main record sets and inspect the first few records.

In [ ]:
# Get list of all record set @id's
record_set_ids = [rs._id for rs in metadata.record_sets]

# Create a DataFrame for each record set
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded record set: {rset_id}, shape: {df.shape}")
        print(f"Columns: {list(df.columns)}\n")
    else:
        print(f"No records for record set: {rset_id}\n")
# For demonstration, display the head of the first table
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Showing first 5 rows for record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping values.

For demonstration, let's pick one loaded record set, attempt to identify a numeric field, and perform filtering and normalization.

In [ ]:
import numpy as np

# Use the first non-empty DataFrame and select a numeric field
if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    # Try to autodetect a numeric field (float or int)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        # Filter records for values > threshold
        threshold = df[numeric_field].quantile(0.75)  # Example: upper quartile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records in '{record_set_id}' where {numeric_field} > {threshold}\n")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}':\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Grouping by the first non-numeric field if present
        group_candidates = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() <= 10]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouped mean by field '{group_field}':\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found for demonstration.")
    else:
        print("No numeric field detected in first record set.")
else:
    print("No record sets loaded.")

## 5. Visualization
Let's visualize the numeric field's distribution and, if possible, group means across a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes and numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If a group field was used, visualize means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric or group fields for visualization.")

## 6. Conclusion
- Loaded Croissant-compliant FAIR^2 dataset metadata and record sets with `mlcroissant`.
- Displayed structure, record set and field `@id`s, loaded data into DataFrames, and applied basic exploratory analyses.
- Filtered, normalized, and visualized numeric distribution and simple group statistics.

You can now build on this notebook to perform deeper statistical or domain-driven analysis, referencing all entities (record sets, fields) by their `@id` as per the FAIR principles and Croissant schema best practices.

**More info:** For schema details visit [Croissant specification](https://mlcommons.org/croissant/spec/).